In [ ]:
from utils.postgres_util import (
    get_engine_from_env,
    read_sql_dataframe,
)

from utils.synthetic.pipeline.bronze_handoff import (
    handoff_final_aligned_observations_to_bronze, 
    run_bronze_handoff_loop,
)

---

In [ ]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

BATCH_SIZE = int(500)

IF_EXISTS_FLAG = str("replace")
COMPLETE_ONLY_FLAG = True
STOP_ON_FAILURE_FLAG = True
MAX_ITERATIONS = None

FINAL_ALIGNMENT_SOURCE_TABLE_NAME = str("synthetic_sensor_observations_final_output")
BRONZE_OUTPUT_TABLE_NAME = str("bronze_observations_input_stage")

---

In [ ]:
engine = get_engine_from_env()

----


### Loop - Run Mode Selection

In [ ]:
# Script style

# Mode

# "row" | "row_batch" | "full_batch"

RUN_MODE = "row_batch"


In [ ]:

results = run_bronze_handoff_loop(
    engine=engine,
    mode=RUN_MODE,          # "row" | "row_batch" | "full_batch"
    batch_size=BATCH_SIZE,
    schema=SCHEMA,
    source_table=FINAL_ALIGNMENT_SOURCE_TABLE_NAME,
    target_table=BRONZE_OUTPUT_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    complete_only=COMPLETE_ONLY_FLAG,
    max_iterations=MAX_ITERATIONS,
    stop_on_failure=STOP_ON_FAILURE_FLAG,
)

print(results[-1] if results else {"status": "no_run"})

----

#### One Shot Full Run

In [ ]:
result = handoff_final_aligned_observations_to_bronze(
    engine=engine,
    mode="full_batch",
    batch_size=500,  # ignored in full_batch
    schema="capstone",
    source_table="synthetic_sensor_observations_final_aligned_stage",
    target_table="bronze_observations_input_stage",
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    complete_only=True,
)

print(result)

---

In [ ]:
# Dispose Engine
engine.dispose()